In [0]:
import pyspark.sql.functions as f
df = spark.read.table('data.orders.orders')
df.groupBy('customer_id').agg(f.count(f.col('order_id')).alias('total')).select('customer_id', 'total').display()

In [0]:
pro = spark.read.table('data.orders.products')
ord = spark.read.table('data.orders.orders')

df = ord.join(pro, ord.product_id == pro.product_id)
df = df.groupBy(pro.product_id).agg(
    f.round(f.sum(ord.quantity * pro.price), 2).alias('rev')
).select(pro.product_id, 'rev')

display(df)

In [0]:
import pyspark.sql.functions as f
df = spark.read.table('data.orders.customers')
df = df.groupBy('city').agg(f.count('customer_id').alias('total'))
df.display()

In [0]:
import pyspark.sql.functions as f
c = spark.read.table('data.orders.customers').alias('c')
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = o.join(c, c.customer_id == o.customer_id)\
    .join(p, o.product_id == p.product_id)\
        .select('o.order_id','c.name','p.product_name')

display(df)


In [0]:
import pyspark.sql.functions as f
c = spark.read.table('data.orders.customers').alias('c')
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = o.join(c, c.customer_id == o.customer_id)\
    .join(p, o.product_id == p.product_id)\
    .withColumn('amount', f.col('quantity')*f.col('price'))\
    .select(f.col('c.name'),f.col('p.product_name'),f.col('p.price'),f.col('o.quantity'),f.col('amount'))
display(df)

In [0]:
import pyspark.sql.functions as f
c = spark.read.table('data.orders.customers').alias('c')
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = o.join(c, c.customer_id == o.customer_id)\
    .join(p, o.product_id == p.product_id)\
    .groupBy(c.name).agg(f.round(f.sum(f.col('quantity')*f.col('price')),2).alias('amount'))
display(df)

In [0]:
import pyspark.sql.functions as f
c = spark.read.table('data.orders.customers').alias('c')
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = o.join(c, c.customer_id == o.customer_id)\
    .join(p, o.product_id == p.product_id)\
    .groupBy(c.name).agg(f.round(f.sum(f.col('quantity')*f.col('price')),2).alias('amount'))\
    .orderBy(f.desc('amount')).limit(5)
display(df)

In [0]:
import pyspark.sql.functions as f
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = o.join(p, o.product_id == p.product_id)\
    .groupBy(p.product_id,p.product_name).agg(f.sum(f.col('o.quantity')).alias('total_sold'))\
    .orderBy(f.desc('total_sold')).limit(3)
display(df)

In [0]:
import pyspark.sql.functions as f
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = o.join(p, o.product_id == p.product_id)\
    .groupBy(p.product_id,p.product_name).agg(f.sum(f.col('o.quantity')*f.col('p.price')).alias('total_amount'))\
    .orderBy(f.desc('total_amount')).limit(1)
display(df)

In [0]:
import pyspark.sql.functions as f
c = spark.read.table('data.orders.customers').alias('c')
o = spark.read.table('data.orders.orders').alias('o')

df = c.join(o, c.customer_id == o.customer_id, how='left' )\
    .where(f.col('o.order_id').isNull())\
    .select('c.name')
display(df)

In [0]:
import pyspark.sql.functions as f
p = spark.read.table('data.orders.products').alias('p')
o = spark.read.table('data.orders.orders').alias('o')

df = p.join(o, o.product_id == p.product_id, how='left' )\
    .where(f.col('o.order_id').isNull())\
    .select('p.product_name')
display(df)

In [0]:
import pyspark.sql.functions as f
o = spark.read.table('data.orders.orders')
df = o.withColumn("date", f.date_format(f.col('order_date'), 'yyyy-MM'))\
    .select('order_id', 'order_date', 'date').show(truncate=False)

In [0]:
import pyspark.sql.functions as f
o = spark.read.table('data.orders.orders').alias('o')
p = spark.read.table('data.orders.products').alias('p')

df = o.join(p, o.product_id == p.product_id)\
    .withColumn("date", f.date_format(f.col('o.order_date'), 'yyyy-MM-dd'))\
    .groupBy("date")\
    .agg(f.round(f.sum(f.col('o.quantity') * f.col('p.price')), 2).alias('daily_rev'))\
    .orderBy("date")

display(df)

In [0]:
import pyspark.sql.functions as f

c = spark.read.table("data.orders.customers")
o = spark.read.table("data.orders.orders")

df = c.join(o, on="customer_id", how="left_anti") \
      .filter(f.col("signup_date") >= f.date_sub(f.current_date(), 30)) \
      .select("name")

display(df)

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.window import Window

c = spark.read.table("data.orders.customers")
o = spark.read.table("data.orders.orders")
p = spark.read.table("data.orders.products")
df = c.join(o, "customer_id").join(p, "product_id")\
      .groupBy("customer_id", "name").agg(f.sum(f.col("quantity") * f.col("price")).alias("amount"))

w = Window.orderBy(f.col("amount").desc())
df = df.withColumn("rank", f.rank().over(w)) \
       .select("name", f.round("amount", 2).alias("amount"), "rank")

display(df)

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.window import Window

c = spark.read.table("data.orders.customers").alias('c')
o = spark.read.table("data.orders.orders").alias('o')
df = c.join(o, "customer_id")\
    .groupBy("c.customer_id", "c.name").agg(f.min('o.order_date').alias("first_date"))\
    .select('c.customer_id','c.name','first_date')
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
o = spark.read.table('data.orders.orders').alias('o')
p = spark.read.table('data.orders.products').alias('p')
c = spark.read.table('data.orders.customers').alias('c')

w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())

df = o.join(c, "customer_id").join(p, "product_id")\
    .withColumn("rn", F.row_number().over(w))\
    .filter(F.col("rn") == 1).select("c.name", "p.product_name", "o.order_date")

display(df)